In [31]:
import re
from dataclasses import dataclass
from typing import List, Optional, Tuple

import numpy as np
from sentence_transformers import SentenceTransformer

Aligned = List[Tuple[Optional[List[str]], Optional[List[str]]]]


# -------------------------
# 1) Sentence splitting
# -------------------------

def split_sentences(text: str) -> List[str]:
    if not text:
        return []

    # normalize spaces (do NOT remove newlines yet)
    text = re.sub(r"[ \t]+", " ", text.strip())

    # split on:
    # 1) sentence-ending punctuation followed by whitespace
    # 2) one or more newlines
    sentences = re.split(r"(?<=[.!?])\s+|\n+", text)

    return [s.strip() for s in sentences if s.strip()]

In [32]:
import re
from dataclasses import dataclass
from typing import List, Optional, Tuple

import numpy as np
from sentence_transformers import SentenceTransformer

Aligned = List[Tuple[Optional[List[str]], Optional[List[str]]]]

# -------------------------
# Helpers
# -------------------------
def _l2_normalize(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    n = np.linalg.norm(x, axis=-1, keepdims=True)
    return x / np.maximum(n, eps)

def _prefix_sums(mat: np.ndarray) -> np.ndarray:
    ps = np.zeros((mat.shape[0] + 1, mat.shape[1]), dtype=np.float32)
    ps[1:] = np.cumsum(mat.astype(np.float32), axis=0)
    return ps

def _avg_block(ps: np.ndarray, start: int, length: int) -> np.ndarray:
    s = ps[start + length] - ps[start]
    return s / float(length)

def _cosine(a: np.ndarray, b: np.ndarray) -> float:
    denom = (np.linalg.norm(a) * np.linalg.norm(b))
    if denom <= 1e-12:
        return 0.0
    return float(np.dot(a, b) / denom)

# -------------------------
# DP structs
# -------------------------
@dataclass
class _Move:
    pi: int
    pj: int
    li: int
    rj: int
    kind: str  # "match", "gapL", "gapR"

@dataclass
class _LocalResult:
    score: float
    end_i: int
    end_j: int
    start_i: int
    start_j: int
    moves: List[_Move]

# -------------------------
# Core: local alignment
# -------------------------
def _local_block_align(
    left: List[str],
    right: List[str],
    left_emb: np.ndarray,
    right_emb: np.ndarray,
    *,
    max_group: int = 2,
    sim_floor: float = 0.55,
    gap_penalty: float = 0.30,
    length_penalty: float = 0.10,

    # NEW: protect strong 1-1 matches from being absorbed into bigger blocks
    protect_strong: bool = True,
    strong_match: float = 0.88,   # cosine threshold for "anchor-like" match
) -> _LocalResult:
    """
    Smith–Waterman style local alignment over sentence blocks.
    Now includes a guard: if a candidate glued block contains a strong 1-1 match,
    we disallow that glued block (forces splitting like you want).
    """
    n, m = len(left), len(right)
    if n == 0 or m == 0:
        return _LocalResult(0.0, 0, 0, 0, 0, [])

    # Precompute single-sentence cosine sims (cheap, sizes are small)
    # left_emb/right_emb are already L2-normalized in your pipeline,
    # so cosine = dot product.
    single_sim = (left_emb @ right_emb.T).astype(np.float32)  # shape (n, m)

    Lps = _prefix_sums(left_emb)
    Rps = _prefix_sums(right_emb)

    dp = np.zeros((n + 1, m + 1), dtype=np.float32)
    back: List[List[Optional[_Move]]] = [[None] * (m + 1) for _ in range(n + 1)]

    best_score = 0.0
    best_pos = (0, 0)

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            best = 0.0
            best_move = None

            # gap from left
            for li in range(1, max_group + 1):
                if i - li < 0:
                    break
                cand = float(dp[i - li, j] - gap_penalty * li)
                if cand > best:
                    best = cand
                    best_move = _Move(i - li, j, li, 0, "gapL")

            # gap from right
            for rj in range(1, max_group + 1):
                if j - rj < 0:
                    break
                cand = float(dp[i, j - rj] - gap_penalty * rj)
                if cand > best:
                    best = cand
                    best_move = _Move(i, j - rj, 0, rj, "gapR")

            # match blocks
            for li in range(1, max_group + 1):
                if i - li < 0:
                    break
                a = _avg_block(Lps, i - li, li)

                for rj in range(1, max_group + 1):
                    if j - rj < 0:
                        break

                    # NEW: if trying to glue (li>1 or rj>1), forbid it when the rectangle
                    # contains a strong 1-1 match. This forces the exact match to stand alone.
                    if protect_strong and (li > 1 or rj > 1):
                        block_max = float(single_sim[i - li : i, j - rj : j].max())
                        if block_max >= strong_match:
                            continue

                    b = _avg_block(Rps, j - rj, rj)
                    sim = _cosine(a, b)

                    block_cost = length_penalty * ((li - 1) + (rj - 1))
                    match_score = (sim - sim_floor) - block_cost

                    cand = float(dp[i - li, j - rj] + match_score)
                    if cand > best:
                        best = cand
                        best_move = _Move(i - li, j - rj, li, rj, "match")

            dp[i, j] = best
            back[i][j] = best_move

            if best > best_score:
                best_score = best
                best_pos = (i, j)

    # Traceback from best_pos until dp==0
    end_i, end_j = best_pos
    moves_rev: List[_Move] = []
    ci, cj = end_i, end_j
    while ci > 0 and cj > 0 and dp[ci, cj] > 1e-8:
        mv = back[ci][cj]
        if mv is None:
            break
        moves_rev.append(mv)
        ci, cj = mv.pi, mv.pj

    moves_rev.reverse()
    start_i, start_j = ci, cj

    return _LocalResult(
        score=float(best_score),
        end_i=end_i,
        end_j=end_j,
        start_i=start_i,
        start_j=start_j,
        moves=moves_rev,
    )

# -------------------------
# Convert moves -> groups
# -------------------------
def _moves_to_groups(
    left: List[str], right: List[str],
    start_i: int, start_j: int,
    moves: List[_Move]
) -> Aligned:
    out: Aligned = []
    i, j = start_i, start_j
    for mv in moves:
        if mv.kind == "match":
            out.append((left[i : i + mv.li], right[j : j + mv.rj]))
            i += mv.li
            j += mv.rj
        elif mv.kind == "gapL":
            out.append((left[i : i + mv.li], None))
            i += mv.li
        elif mv.kind == "gapR":
            out.append((None, right[j : j + mv.rj]))
            j += mv.rj
    return out

def _all_gaps(left: List[str], right: List[str]) -> Aligned:
    out: Aligned = []
    for s in left:
        out.append(([s], None))
    for s in right:
        out.append((None, [s]))
    return out

# -------------------------
# Top-level wrapper
# -------------------------
def local_align_texts(
    model: SentenceTransformer,
    left_text: str,
    right_text: str,
    *,
    max_group: int = 1,          # RECOMMEND: 1 for your instruction data
    sim_floor: float = 0.6,
    gap_penalty: float = 0.30,
    length_penalty: float = 0.10,
    min_segment_score: float = 0.10,
    protect_strong: bool = True,
    strong_match: float = 0.85,
) -> Aligned:
    left_sents = split_into_sentences(left_text)
    right_sents = split_into_sentences(right_text)

    L = _l2_normalize(np.asarray(model.encode(left_sents, batch_size=32, convert_to_numpy=True), dtype=np.float32))
    R = _l2_normalize(np.asarray(model.encode(right_sents, batch_size=32, convert_to_numpy=True), dtype=np.float32))

    def rec(li0: int, li1: int, rj0: int, rj1: int) -> Aligned:
        Ls = left_sents[li0:li1]
        Rs = right_sents[rj0:rj1]
        Le = L[li0:li1]
        Re = R[rj0:rj1]

        if not Ls or not Rs:
            return _all_gaps(Ls, Rs)

        res = _local_block_align(
            Ls, Rs, Le, Re,
            max_group=max_group,
            sim_floor=sim_floor,
            gap_penalty=gap_penalty,
            length_penalty=length_penalty,
            protect_strong=protect_strong,
            strong_match=strong_match,
        )

        if res.score < min_segment_score or not res.moves:
            return _all_gaps(Ls, Rs)

        pre = rec(li0, li0 + res.start_i, rj0, rj0 + res.start_j)
        mid = _moves_to_groups(Ls, Rs, res.start_i, res.start_j, res.moves)
        post = rec(li0 + res.end_i, li1, rj0 + res.end_j, rj1)
        return pre + mid + post

    return rec(0, len(left_sents), 0, len(right_sents))

In [5]:
model_name: str = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_name, device = "cuda:0")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1041.75it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [39]:
# -------------------------
# Example
# -------------------------
if __name__ == "__main__":
    left = """You are given a relation name, a description of the relation in brackets, a support sentence that exemplifies the relation, and a query sentence.

A relation connects the Subject and the Object entities. The Subject and the Object entities are indicated with subject and object tags, respectively. You need to decide whether the relation holds between the Subject and the Object in the query sentence.

If the relation holds between the Subject and Object in the query sentence, answer "yes"; otherwise, answer "no".
"""

    right = """You are given a relation name, a description of the relation in brackets, a support sentence that exemplifies the relation, and a query sentence. The subject and object entities are marked with "subject" and "object" tags, respectively. Your task is to determine whether the query sentence explicitly or implicitly expresses the relation defined in the description between the subject and object entities. 

Focus strictly on the subject and object entities marked with the tags. Ignore other entities or information in the sentence. Apply the relation's definition to the query sentence: does the relation hold between the subject and object as described? Answer "yes" if the relation is clearly expressed, "no" otherwise.
"""

    # groups = local_align_texts(model, left, right)
    groups = local_align_texts(model,
        left, right,
        max_group=3,          # keep many-to-many
        length_penalty=0.15,  # was ~0.03; raise to discourage 2↔1, 3↔1, etc.
        sim_floor = 0.55
    )
    for k, (Lgrp, Rgrp) in enumerate(groups, 1):
        print(f"\nGroup {k}:")
        print("  L:", Lgrp)
        print("  R:", Rgrp)


Group 1:
  L: ['You are given a relation name, a description of the relation in brackets, a support sentence that exemplifies the relation, and a query sentence.']
  R: ['You are given a relation name, a description of the relation in brackets, a support sentence that exemplifies the relation, and a query sentence.']

Group 2:
  L: ['A relation connects the Subject and the Object entities.']
  R: None

Group 3:
  L: ['The Subject and the Object entities are indicated with subject and object tags, respectively.']
  R: ['The subject and object entities are marked with "subject" and "object" tags, respectively.']

Group 4:
  L: ['You need to decide whether the relation holds between the Subject and the Object in the query sentence.']
  R: ['Your task is to determine whether the query sentence explicitly or implicitly expresses the relation defined in the description between the subject and object entities.']

Group 5:
  L: None
  R: ['Focus strictly on the subject and object entities mar